# 00 — SmarTRIZ Dataset Analysis

Loads all dataset files, prints schema, visualizes distributions, cleans data, and saves `training_dataset_clean.json`. Run this notebook first before any training.

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'
JUDGE_SCORE_MIN = 7.0
MAX_TOKENS = 4096
TOKENIZER_NAME = 'Qwen/Qwen2.5-7B-Instruct'
# ──────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers tqdm matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os
from pathlib import Path

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def load_training(path):
    """Handles both JSON array and JSON object with .cases wrapper."""
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and 'cases' in data:
        return data['cases']
    raise ValueError(f'Unexpected training_dataset format: {list(data.keys())}')

training = load_training(f'{DRIVE_PATH}data/training_dataset.json')
rejected = load_jsonl(f'{DRIVE_PATH}data/rejected_dataset.jsonl')
borderline = load_jsonl(f'{DRIVE_PATH}data/borderline.jsonl') if Path(f'{DRIVE_PATH}data/borderline.jsonl').exists() else []

print(f'training:   {len(training)} records')
print(f'rejected:   {len(rejected)} records')
print(f'borderline: {len(borderline)} records')

In [ ]:
# ── Schema inspection
def print_schema(record, prefix=''):
    for k, v in record.items():
        t = type(v).__name__
        if isinstance(v, dict):
            print(f'{prefix}{k}: dict')
            print_schema(v, prefix + '  ')
        elif isinstance(v, list):
            inner = type(v[0]).__name__ if v else 'empty'
            print(f'{prefix}{k}: list[{inner}] (len={len(v)})')
        else:
            preview = str(v)[:60].replace('\n', ' ')
            print(f'{prefix}{k}: {t} = {preview!r}')

print('=== training_dataset.json case schema ===')
print_schema(training[0])
print('\n=== rejected_dataset.jsonl record schema ===')
print_schema(rejected[0])

In [ ]:
import matplotlib.pyplot as plt
import collections

def count_field(records, field_path):
    counts = collections.Counter()
    for r in records:
        v = r
        for k in field_path.split('.'):
            v = v.get(k, None) if isinstance(v, dict) else None
        counts[str(v)] += 1
    return counts

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SmarTRIZ Dataset Distributions', fontsize=16)

for ax, (title, field) in zip(axes.flat, [
    ('Complexity', 'complexity'),
    ('Generation Method', 'meta.generation_method'),
    ('Top 10 Domains', 'domain'),
    ('Judge Average Score', 'meta.judge_scores.average'),
]):
    counts = count_field(training, field)
    top = dict(counts.most_common(10))
    ax.bar(list(top.keys()), list(top.values()))
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
os.makedirs(f'{DRIVE_PATH}data', exist_ok=True)
plt.savefig(f'{DRIVE_PATH}data/dataset_distributions.png', dpi=150)
plt.show()
print('Saved distribution plot to DRIVE_PATH/data/dataset_distributions.png')

In [ ]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, trust_remote_code=True)

def count_tokens(record):
    text = ' '.join([
        record.get('problem', ''),
        record.get('reasoning_chain', ''),
        record.get('solution', ''),
    ])
    return len(tokenizer.encode(text, add_special_tokens=False))

for r in tqdm(training, desc='counting tokens'):
    r['_token_count'] = count_tokens(r)

over_limit = [r for r in training if r['_token_count'] > MAX_TOKENS]
all_counts = [r['_token_count'] for r in training]
print(f'Examples over {MAX_TOKENS} tokens: {len(over_limit)} ({len(over_limit)/len(training)*100:.1f}%)')
print(f'Token stats: min={min(all_counts)} median={sorted(all_counts)[len(all_counts)//2]} max={max(all_counts)}')

In [ ]:
# ── Clean dataset: remove over-limit, low-score, malformed
def is_valid(r):
    required = ['problem', 'reasoning_chain', 'solution', 'contradiction_pair', 'inventive_principles']
    if any(not r.get(f) for f in required):
        return False, 'missing_field'
    cp = r.get('contradiction_pair', {})
    if not cp.get('improving_parameter') or not cp.get('worsening_parameter'):
        return False, 'bad_contradiction_pair'
    if not isinstance(r.get('inventive_principles'), list) or len(r['inventive_principles']) == 0:
        return False, 'empty_principles'
    score = r.get('meta', {}).get('judge_scores', {}) or {}
    average = score.get('average')
    if average is not None and float(average) < JUDGE_SCORE_MIN:
        return False, 'low_judge_score'
    if r.get('_token_count', 0) > MAX_TOKENS:
        return False, 'over_token_limit'
    return True, None

clean, removed = [], []
for r in training:
    ok, reason = is_valid(r)
    if ok:
        clean.append(r)
    else:
        removed.append({'id': r.get('id'), 'reason': reason})

print(f'Clean examples: {len(clean)}')
print(f'Removed:        {len(removed)}')
if removed:
    from collections import Counter
    print('Removal reasons:', Counter(r["reason"] for r in removed))

In [ ]:
# ── Save cleaned train + test split
import random

for r in clean:
    r.pop('_token_count', None)

random.seed(42)
random.shuffle(clean)
split = int(len(clean) * 0.9)
train_split = clean[:split]
test_split  = clean[split:]

out_train = f'{DRIVE_PATH}data/training_dataset_clean.json'
out_test  = f'{DRIVE_PATH}data/test_split.json'

with open(out_train, 'w') as f:
    json.dump({'dataset_name': 'SmarTRIZ-Synthetic-v1-clean',
               'total_cases': len(train_split),
               'cases': train_split}, f, indent=2)

with open(out_test, 'w') as f:
    json.dump({'cases': test_split}, f, indent=2)

print(f'Saved {len(train_split)} train examples → {out_train}')
print(f'Saved {len(test_split)} test examples  → {out_test}')
print('\nReady for Notebook 01 (SFT training)')